# 01 · 张量与自动微分 Tensors & Autograd

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解 PyTorch **张量**（Tensor）与 NumPy 数组的异同，以及设备（device）的概念。
2. 用 `requires_grad=True` 标记需要计算梯度的张量，构建**计算图**（computation graph）。
3. 调用 `.backward()` 触发反向传播（backpropagation），从 `.grad` 读取梯度。
4. 用 `torch.no_grad()` 上下文管理器关闭梯度跟踪（推理 / inference 阶段）。
5. 理解"PyTorch 就是自动化的 nanograd"——`Tensor` 与 n1~n3 中的 `Value` 遵循完全相同的计算图原理。

> 对应能力：**SH8**


## 概念讲解 Concepts

### 1. 张量 Tensor

PyTorch 的核心数据结构是 **张量**（Tensor）——n 维数组，与 `numpy.ndarray` 功能相似，
但额外支持 GPU 加速和自动微分。

```python
import torch
t = torch.tensor([1.0, 2.0, 3.0])   # 1D 张量
m = torch.zeros(3, 4)                # 3×4 全零矩阵
```

张量与 NumPy 可以互转：`t.numpy()` / `torch.from_numpy(arr)`。

---

### 2. 计算图与 requires_grad

每当对带有 `requires_grad=True` 的张量执行运算，PyTorch 就会自动记录一个**计算图**（computation graph）。
这与 n1 中的 `Value._prev` 机制完全对应：

| nanograd (`Value`) | PyTorch (`Tensor`) |
|---|---|
| `v._prev` | 隐式计算图 |
| `v.grad` | `t.grad` |
| `v.backward()` | `t.backward()` |
| 手写 `_backward` | PyTorch 内部 C++ 实现 |

"PyTorch 就是自动化的 nanograd" — 当你调用 `.backward()`，
PyTorch 做的事情与 n1/n2 中的拓扑排序反向传播**完全相同**。

---

### 3. .backward() 与 .grad

```python
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = (x ** 2).sum()   # 标量
y.backward()          # 触发反向传播
print(x.grad)         # tensor([2., 4.]) — dy/dx = 2x
```

注意：`.backward()` 需要从**标量**（scalar）输出调用；
若输出是向量/矩阵，需传入 `gradient` 参数作为初始梯度（向量-雅可比积）。

---

### 4. torch.no_grad()

推理（inference）阶段不需要梯度，用 `torch.no_grad()` 可节省内存和计算：

```python
with torch.no_grad():
    pred = model(x)    # 不构建计算图，更快
```

---

### 5. 梯度累积与 .grad 清零

PyTorch 的 `.grad` 使用**累加**（`+=`）语义，与 nanograd 中的 `value.grad += ...` 一致。
因此训练循环中每次迭代前必须清零：

```python
optimizer.zero_grad()   # 清零所有参数的 .grad
loss.backward()
optimizer.step()
```


## 示例 Worked Example — `grad_of_square_sum`

In [ ]:
import torch

torch.manual_seed(42)

# ── 基本张量操作 Basic Tensor Operations ──────────────────────────────────
x = torch.tensor([1.0, 2.0, 3.0])
print("x =", x)
print("x.shape =", x.shape, " | x.dtype =", x.dtype)
print("x + 1 =", x + 1)
print("x ** 2 =", x ** 2)
print()

# ── requires_grad 示例 ───────────────────────────────────────────────────
a = torch.tensor([2.0, 3.0], requires_grad=True)
b = (a ** 2).sum()         # b = a[0]^2 + a[1]^2 = 4 + 9 = 13
print(f"a = {a}")
print(f"b = sum(a^2) = {b.item():.1f}  (应为 13.0)")
b.backward()
print(f"a.grad = {a.grad}  (应为 [4., 6.] = 2*a)")


In [ ]:
# ── 镜像练习：grad_of_square_sum ─────────────────────────────────────────
# 与 w6-torch-basics 练习完全对应

import torch

def grad_of_square_sum(x):
    # 给定 1D 张量 x，计算 y = sum(x**2) 对 x 的梯度（返回张量，应等于 2x）
    x = torch.as_tensor(x, dtype=torch.float32).clone().requires_grad_(True)
    y = (x ** 2).sum()
    y.backward()
    return x.grad

# 验证
test_x = torch.tensor([1.0, 2.0, 3.0, 4.0])
g = grad_of_square_sum(test_x)
print(f"x       = {test_x.tolist()}")
print(f"grad    = {g.tolist()}  (应为 2x = {(2*test_x).tolist()})")
assert torch.allclose(g, 2 * test_x), "梯度不等于 2x"
print("✓ grad_of_square_sum 验证通过")
print()

# nanograd 类比（回顾）
print("nanograd 类比 Nanograd analogy:")
print("  Value.grad += ...         -> tensor.grad (累加语义)")
print("  d.backward()              -> y.backward()  (完全相同！)")
print("  p.data -= lr * p.grad    -> optimizer.step()")


In [ ]:
# ── torch.no_grad() 演示 ─────────────────────────────────────────────────
import torch

torch.manual_seed(0)
x = torch.tensor([1.0, 2.0], requires_grad=True)

# 有梯度模式：构建计算图
y_with_grad = (x ** 2).sum()
print(f"y_with_grad.requires_grad = {y_with_grad.requires_grad}")  # True

# 无梯度模式：不构建计算图（推理/评估时使用）
with torch.no_grad():
    y_no_grad = (x ** 2).sum()
    print(f"y_no_grad.requires_grad  = {y_no_grad.requires_grad}")   # False

print()
print("推理阶段用 torch.no_grad()：省内存、提速，不跟踪梯度。")

# ── 叶节点与中间节点 Leaf vs. intermediate ────────────────────────────────
a = torch.tensor(2.0, requires_grad=True)
b = a * 3
c = b ** 2
c.backward()
print()
print(f"a = {a.item()},  a.grad = {a.grad.item():.1f}")
# dc/da = 2*b * db/da = 2*(3a)*3 = 18a = 18*2 = 36
print(f"  dc/da = 2*(3a)*3 = 18*a = 18*{a.item():.0f} = {18*a.item():.1f}  check OK")


## 动手 Exercise

In [ ]:
# ── 动手练习：更复杂表达式的梯度 ──────────────────────────────────────────
# f(x, w) = sum( (w*x - y)^2 )  （均方误差）
# 其中 x = [1,2,3], y = [2,4,6]（即 y = 2*x），w = 1.5
# 手工推导：
#   f = sum((w*xi - 2*xi)^2) = (w-2)^2 * sum(xi^2) = (w-2)^2 * 14
#   df/dw = 2*(w-2) * 14 = 28*(w-2)
#   在 w=1.5：df/dw = 28*(1.5-2) = -14.0

import torch

torch.manual_seed(1)

# 使用 float64 确保数值精度
x = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
y_target = 2 * x
w = torch.tensor(1.5, dtype=torch.float64, requires_grad=True)

pred = w * x
loss = ((pred - y_target) ** 2).sum()
loss.backward()

print(f"w = {w.item()},  x = {x.tolist()}")
print(f"loss = {loss.item():.4f}")
print(f"w.grad = {w.grad.item():.4f}  (手工推导应为 {28*(1.5-2):.4f})")

expected = 28 * (1.5 - 2)
assert abs(w.grad.item() - expected) < 1e-6, f"梯度错误: {w.grad.item()} != {expected}"
print("✓ 梯度验证通过")

# 有限差分验证（float64 精度更高）
h = 1e-5
w_p = torch.tensor(1.5 + h, dtype=torch.float64)
w_m = torch.tensor(1.5 - h, dtype=torch.float64)
loss_p = ((w_p * x - y_target)**2).sum().item()
loss_m = ((w_m * x - y_target)**2).sum().item()
fd = (loss_p - loss_m) / (2 * h)
print(f"有限差分梯度: {fd:.6f}  (应与 w.grad 一致)")
assert abs(w.grad.item() - fd) < 1e-6, f"有限差分不一致: fd={fd:.6f} vs grad={w.grad.item():.6f}"
print("✓ 有限差分验证通过")


## 延伸阅读 Further Reading

1. **PyTorch Autograd Mechanics**：<https://pytorch.org/docs/stable/notes/autograd.html> — 官方文档，工业级 autograd 的实现细节。
2. **Karpathy «micrograd»**：<https://github.com/karpathy/micrograd> — 本课 nanograd 的原型，仅 150 行 Python。
3. **PyTorch 入门教程**：<https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html> — 张量操作全览。
4. **《Deep Learning with PyTorch》** Stevens et al. 第 2-3 章 — 张量与 autograd 的系统讲解。


## 关联练习 Related Assignments

```bash
python check.py 01-torch-basics
```

> 实现 `grad_of_square_sum(x)`：对 1D 张量 x 计算 y = sum(x²) 对 x 的梯度（应等于 2x）。

> 能力标签：**SH8** · threshold ≥ 0.7
